# Sesle konuşmacı tanıma

Bu not defteri, yalnızca açık rızasıyla kayıt edilmiş kişiler arasından ses sahibini tahmin etmek içindir. Ses biyometrik veridir; izinsiz kayıt veya takip için kullanmayın. Sonuçlar kesin kimlik doğrulaması değildir.

## Veri yapısı

Bilgisayarında aşağıdaki yapıda bir `dataset.zip` hazırla ve sonraki hücrede yükle. Her kişi için en az 8, tercihen 20+ adet 3–10 saniyelik WAV kaydı kullan.

```text
data/
  ayse/001.wav
  ayse/002.wav
  mehmet/001.wav
```

In [ ]:
!apt-get -qq update && apt-get -qq install -y ffmpeg
!pip -q install speechbrain torchaudio scikit-learn joblib

import os
import zipfile
from pathlib import Path
import joblib
import numpy as np
import torch
import torchaudio
from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
# dataset.zip dosyanı seçip yükle.
uploaded = files.upload()
zip_name = next((name for name in uploaded if name.lower().endswith('.zip')), None)
if zip_name is None:
    raise ValueError('Lütfen data klasörünü içeren bir .zip dosyası yükle.')

with zipfile.ZipFile(zip_name, 'r') as zf:
    zf.extractall('/content')

DATA_DIR = Path('/content/data')
if not DATA_DIR.exists():
    raise FileNotFoundError('Zip içinde data/ klasörü bulunamadı.')

audio_files = sorted([p for p in DATA_DIR.rglob('*') if p.suffix.lower() in {'.wav', '.flac', '.mp3', '.ogg', '.m4a'}])
print(f'{len(audio_files)} ses dosyası bulundu.')
print('Kişiler:', sorted({p.parent.name for p in audio_files}))

In [ ]:
# Önceden eğitilmiş ECAPA-TDNN modeli: ses kaydını sayısal ses imzasına dönüştürür.
from speechbrain.inference.classifiers import EncoderClassifier

device = 'cuda' if torch.cuda.is_available() else 'cpu'
encoder = EncoderClassifier.from_hparams(
    source='speechbrain/spkrec-ecapa-voxceleb',
    savedir='/content/pretrained_spkrec',
    run_opts={'device': device}
)
TARGET_SR = 16000

def load_audio(path):
    waveform, sample_rate = torchaudio.load(path)
    waveform = waveform.mean(dim=0, keepdim=True)  # mono
    if sample_rate != TARGET_SR:
        waveform = torchaudio.functional.resample(waveform, sample_rate, TARGET_SR)
    return waveform

@torch.inference_mode()
def embedding_for(path):
    waveform = load_audio(str(path)).to(device)
    return encoder.encode_batch(waveform).squeeze().cpu().numpy()

In [ ]:
# Ses imzalarını çıkar. Bu işlem veri büyüklüğüne göre birkaç dakika sürebilir.
X, y = [], []
for i, path in enumerate(audio_files, start=1):
    try:
        X.append(embedding_for(path))
        y.append(path.parent.name)
    except Exception as exc:
        print(f'Atlandı: {path.name} — {exc}')
    if i % 10 == 0:
        print(f'{i}/{len(audio_files)} işlendi')

X = np.asarray(X)
y = np.asarray(y)
labels, counts = np.unique(y, return_counts=True)
print(dict(zip(labels, counts)))
if len(labels) < 2 or counts.min() < 4:
    raise ValueError('En az iki kişi ve her kişi için en az dört geçerli kayıt gerekir.')

In [ ]:
# Eğitim ve test ayrımı: her kişiden bazı kayıtlar daha önce görülmeden test edilir.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y
)
model = make_pipeline(
    StandardScaler(),
    SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=SEED)
)
model.fit(X_train, y_train)
pred = model.predict(X_test)
print(f'Test doğruluğu: %{accuracy_score(y_test, pred) * 100:.1f}')
print(classification_report(y_test, pred, zero_division=0))

joblib.dump({'model': model, 'labels': model.classes_.tolist()}, 'speaker_model.joblib')
files.download('speaker_model.joblib')

In [ ]:
# Yeni bir ses dosyası yükleyip tahmin yap. WAV formatı önerilir.
uploaded = files.upload()
test_file = next(iter(uploaded))
test_embedding = embedding_for(test_file).reshape(1, -1)
probabilities = model.predict_proba(test_embedding)[0]
best_index = probabilities.argmax()
print(f'Tahmin: {model.classes_[best_index]}')
print(f'Güven: %{probabilities[best_index] * 100:.1f}')
print('Tüm skorlar:')
for label, score in sorted(zip(model.classes_, probabilities), key=lambda item: item[1], reverse=True):
    print(f'  {label}: %{score * 100:.1f}')

# Düşük güven, modelin bu kişiyi tanımadığını veya kayıt koşullarının farklı olduğunu gösterebilir.